# Train Arm B (blt_entropy_patching) on Kaggle - full pipeline test

See `plans/PLAN.md` (question 1.3, Arm B), `phases/phase-2-small-train.md`.

Runs the ACTUAL repo code (`vislm/train.py`, `vislm/backbones/models/blt_lm.py`) via the
`nguyennn263/vislm-research-code` dataset mounted at `/kaggle/input/datasets/nguyennn263/vislm-research-code` - same pattern as the
Arm A debug run. Same GPU-detection + torch pin as Arm A (P100 -> torch==2.7.1+cu126).


In [ ]:
import subprocess

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu_name or "(none detected)")

if "P100" in gpu_name:
    print("P100 detected - pinning torch==2.7.1+cu126 (last version supporting sm_60)")
    subprocess.run(
        ["pip", "install", "-q", "torch==2.7.1", "--index-url",
         "https://download.pytorch.org/whl/cu126"],
        check=True,
    )
else:
    print("Not a P100 - keeping the pre-installed torch build")

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())


In [ ]:
import os

_candidates = [
    "/kaggle/input/vislm-research-code",
    "/kaggle/input/datasets/nguyennn263/vislm-research-code",
]
CODE_DIR = next(p for p in _candidates if os.path.isdir(p))
os.environ["PYTHONPATH"] = CODE_DIR
print("CODE_DIR:", CODE_DIR)

!pip install -q "transformers==4.46.3" datasets pyyaml


In [ ]:
!python {CODE_DIR}/setup/download_prepare_data.py \
  --target-gb 0.05 --out-dir /kaggle/working/data/prepared/fineweb2_vi


In [ ]:
# Rerun after fixing entropy_threshold (3.7 -> 3.0, see phases/phase-2-small-train.md) -
# Arm B's per-patch Python loop (see blt_lm.py docstring) is not vectorized like
# Arm A - start small to measure real per-step time before committing to 300 steps.
!python -m vislm.train {CODE_DIR}/pillar1_configs/1_3_arm_B_blt.yaml \
  dataset=/kaggle/working/data/prepared/fineweb2_vi \
  run_dir=/kaggle/working/runs/arm_B_debug \
  model.max_seq_len=128 train.batch_size=4 \
  train.max_steps=30 train.entropy_pretrain_steps=20


In [ ]:
import json

losses = []
with open("/kaggle/working/runs/arm_B_debug/metrics.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if "step" in row:
            losses.append(row["loss"])

summary = {
    "n_steps": len(losses),
    "first_loss": losses[0],
    "last_loss": losses[-1],
    "min_loss": min(losses),
}
print(summary)

with open("/kaggle/working/metrics_train_arm_b_debug.jsonl", "w") as f:
    f.write(json.dumps({"section": "train_arm_b_debug", "results": summary}) + "\n")
